# 🧬 Evolutionary Design: Genotype to Phenotype
This notebook demonstrates the **Ribosome** concept in an Evolutionary Outer Loop. 

### The Core Concepts:
1. **Genotype**: The search space (a simple list of design decisions).
2. **Phenotype**: The actual Keras model built from those decisions.
3. **Evaluation**: Training the phenotype on the Jena Climate data to get a **Fitness Score**.

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np

print("TensorFlow version:", tf.__version__)

I0000 00:00:1774288279.986518    1047 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774288280.014671    1047 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1774288282.460012    1047 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow version: 2.21.0


## 1. The Ribosome (Mapping Function)
This function translates the **Genotype** (DNA) into the **Phenotype** (Model).

In [2]:
def build_phenotype(genotype, input_shape=(24, 7)):
    """
    Translates a Genotype list into a Keras Phenotype.
    Genotype: [model_type_idx, units, num_layers, dropout_rate, act_idx]
    """
    m_type_idx, units, n_layers, dropout, act_idx = genotype
    
    # Mapping discrete choices
    rnn_class = layers.LSTM if m_type_idx == 0 else layers.GRU
    activation = 'tanh' if act_idx == 0 else 'relu'
    
    model = models.Sequential()
    
    # Build Hidden Layers
    for i in range(n_layers):
        is_last_rnn = (i == n_layers - 1)
        model.add(rnn_class(
            units, 
            activation=activation,
            return_sequences=not is_last_rnn,
            input_shape=input_shape if i == 0 else None
        ))
        model.add(layers.Dropout(dropout))
    
    # Output Layer (Predicting Temperature)
    model.add(layers.Dense(1))
    model.compile(optimizer='adam', loss='mse')
    
    return model

## 2. Running a Generation
We simulate an Evolutionary Algorithm selecting and building two different individuals.

In [3]:
# Define two different 'Genotypes'
population = [
    [0, 64, 2, 0.2, 0], # Individual A: 2-layer LSTM
    [1, 128, 1, 0.1, 1] # Individual B: 1-layer GRU
]

for i, genotype in enumerate(population):
    print(f"\n--- Individual {i} ---")
    print(f"Genotype (DNA): {genotype}")
    
    # Translation to Phenotype
    phenotype = build_phenotype(genotype)
    
    # View the Phenotype structure
    phenotype.summary()
    
    # In a real run, you'd train here:
    # phenotype.fit(x_train, y_train, epochs=3)
    print(f"Phenotype {i} is ready for evaluation.")


--- Individual 0 ---
Genotype (DNA): [0, 64, 2, 0.2, 0]


W0000 00:00:1774288284.635357    1047 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...
/app/data/environments/python/3.12/venv_taap_p4/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 24, 64)         │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 24, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 51,521 (201.25 KB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 128)            │        52,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 52,737 (206.00 KB)

 Trainable params: 52,737 (206.00 KB)

 Non-trainable params: 0 (0.00 B)

Phenotype 1 is ready for evaluation.
